# 🧠 EMS741 Exam Prep: Ollama & Study Environment
This notebook installs Ollama natively into your Conda/Mamba environment (with GPU acceleration), starts the server, and provides a structured workspace for your bookclub-style exam focusing on Lagergren (2020) and Li (2020).

In [8]:
# 1. Install dependencies natively via Conda/Pip (Forcing CUDA/GPU variant)
!mamba install -y -c conda-forge ollama "ollama=*=*cuda*"
!pip install ollama -q

import subprocess
import time
import ollama

# 2. Start the background server
print("Starting GPU-enabled Ollama server...")
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# 3. Pull the model
MODEL_NAME = 'llama3'
print(f"Pulling {MODEL_NAME}...")
!ollama pull {MODEL_NAME}
print("✅ Setup Complete!")


Looking for: ['ollama', 'ollama=[build=*cuda*]']

[+] 0.0s
conda-forge/linux-64                                          No change
[+] 0.1s
conda-forge/noarch ━━━━━━━━━━━╸━━━━━━━━━━━━━   0.0 B /  ??.?MB @  ??.?MB/s  0.1s[+] 0.2s
conda-forge/noarch ━━━━━━━━━━━━━╸━━━━━━━━━━━  14.8MB /  26.2MB @  80.4MB/s  0.2s[+] 0.3s
conda-forge/noarch ━━━━━━━━━━━━━━━━━━━╸━━━━━  21.3MB /  26.2MB @  90.6MB/s  0.3sconda-forge/noarch                                  26.2MB @  97.1MB/s  0.4s

Pinned packages:
  - python 3.11.*


Transaction

  Prefix: /home/jovyan/env/mamba-gpu

  All requested packages already installed

Starting GPU-enabled Ollama server...
Pulling llama3...
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 6a0746a1ec1a: 100% ▕██████████████████▏ 4.7 GB                         
pulling 4fa551d4f938: 100% ▕██████████████████▏  12 KB                         
pulling 8ab4849b038c: 100% ▕██████████████████▏  254 B                   

In [2]:
def query_exam_tutor(prompt, system_context="You are an expert AI tutor helping a robotics MSc student prepare for their EMS741 exam."):
    """Streams the answer from Ollama using the official python client."""
    print(f"\n📝 Question: {prompt}\n")
    print("🤖 Answer:")
    
    response = ollama.chat(model=MODEL_NAME, messages=[
        {'role': 'system', 'content': system_context},
        {'role': 'user', 'content': prompt}
    ], stream=True)
    
    for chunk in response:
        print(chunk['message']['content'], end='', flush=True)
    print("\n\n" + "━"*60)


Loading Context

In [16]:
def load_context(filepath):
    with open(filepath, 'r') as file:
        return file.read()

# Load the dense markdown file you created
my_custom_context = load_context('Ollama-Course-Context-Pack.md')

query_exam_tutor(
    "How does the constraint loss in BINNs enforce biological priors?", 
    system_context=my_custom_context
)


📝 Question: How does the constraint loss in BINNs enforce biological priors?

🤖 Answer:
In the Biologically-Informed Neural Networks (BINNs) paper by Lagergren (2020), the Constraint Loss (L_Constr) is used to enforce biological priors on the learned diffusion and growth rates. Specifically, the constraints are:

1. **Diffusivity must not decrease with density**: ∂D/∂u ≥ 0
2. **Growth must not increase with density**: ∂G/∂u ≤ 0

To enforce these constraints, the authors use a regularization term in the loss function. The Constraint Loss (L_Constr) is calculated as:

L_Constr = λ * (max(0, -∂D/∂u) + max(0, ∂G/∂u))

where λ is a hyperparameter that controls the strength of the regularization.

The key idea is to add a penalty term to the loss function if either of these constraints is violated. The maximum function (`max(0, -∂D/∂u)`) ensures that only violations are penalized, while the `λ` hyperparameter adjusts the weight of this constraint in the overall loss.

When the model is trai

## 🧪 Paper 1: Lagergren 2020 (BINNs)
**Core concepts:** Physics-informed loss, learning unknown PDE terms (diffusivity/growth), scratch-assays, time-delay term, constraints/priors.

In [17]:
ctx_lagergren = """
Context: Lagergren 2020 'Biologically-informed neural networks'.
BINNs learn unknown reaction-diffusion PDE terms directly from sparse scratch-assay data. 
They use an MLP surrogate for density u(x,t), and MLPs for parameters D(u) and G(u). 
They added a time-delay term T(t) to handle initial confluency discrepancy.
"""

query_exam_tutor(
    "Describe the main building blocks of the BINN model (inputs, outputs, architecture, loss function).", 
    system_context=ctx_lagergren
)


📝 Question: Describe the main building blocks of the BINN model (inputs, outputs, architecture, loss function).

🤖 Answer:
The main building blocks of the BINN model are:

**Inputs:**

* Scratch-assay data: a set of sparse measurements of cell density u(x,t) at different spatial locations x and time points t.
* Spatial coordinates x: the location of each measurement in the spatial domain.

**Outputs:**

* Learned reaction-diffusion PDE terms: The model learns the unknown terms D(u) (diffusion coefficient) and G(u) (reaction term) that describe the dynamics of the cell density u(x,t).
* Density predictions: The model predicts the density u(x,t) at new spatial locations x and time points t.

**Architecture:**

* Multilayer Perceptron (MLP): The model uses three MLPs to learn the unknown terms:
	1. A surrogate MLP for density u(x,t), which takes the spatial coordinates x as input and predicts the cell density.
	2. An MLP for the diffusion coefficient D(u), which takes the predicted densi

## 🧪 Paper 2: Li 2020 (CNN Surrogate)
**Core concepts:** Encoder-decoder CNN, 2D reaction-diffusion, purely data-driven (trained on 1M FEM samples), no physics in loss, ~300x speedup.

In [18]:
ctx_li = """
Context: Li 2020 'CNN surrogate for reaction-diffusion'.
A CNN encoder-decoder predicts 2D concentration fields bypassing FEM.
Inputs: 4-channel tensor (geometry/BC, D, K, time) as 21x21 matrices.
Data-driven (MSE loss on synthetic FEM data), no physical constraints in loss.
"""

query_exam_tutor(
    "What are the inputs, outputs, and architecture of this CNN model? How does it handle time-dependence?", 
    system_context=ctx_li
)


📝 Question: What are the inputs, outputs, and architecture of this CNN model? How does it handle time-dependence?

🤖 Answer:
According to the Li et al. (2020) paper "CNN surrogate for reaction-diffusion", the CNN model takes 4-channel tensor as input and predicts 2D concentration fields.

**Inputs:**

The input is a 4-channel tensor, where each channel represents:

1. Geometry/BC (boundary conditions): A 21x21 matrix representing the spatial distribution of geometry or boundary conditions.
2. D (diffusion coefficient): A 21x21 matrix representing the spatial distribution of the diffusion coefficient.
3. K (reaction rate constant): A 21x21 matrix representing the spatial distribution of the reaction rate constant.
4. Time: A scalar value representing the time step.

**Outputs:**

The output is a 2D concentration field, which represents the spatial distribution of the concentration at each time step.

**Architecture:**

The CNN model consists of an encoder-decoder architecture:

1. **En

## ⚖️ Cross-Paper Comparison
**Core concepts:** Physics-informed (BINNs) vs Physics-based training data (Li CNN).

In [19]:
ctx_compare = """
You are comparing BINNs (Lagergren, 2020) and CNN surrogates (Li, 2020).
Lagergren: Physics in the loss function, continuous MLP, sparse real data.
Li: Physics only in the training data generation, discrete 2D CNN, massive synthetic data.
"""

query_exam_tutor(
    "Compare how these two papers incorporate physical laws. Which one enforces physical constraints during training, and how?", 
    system_context=ctx_compare
)


📝 Question: Compare how these two papers incorporate physical laws. Which one enforces physical constraints during training, and how?

🤖 Answer:
The key difference between BINNs (Lagergren, 2020) and CNN surrogates (Li, 2020) is how they incorporate physical laws:

**BINNs (Lagergren, 2020)**: In this approach, the authors enforce physical constraints during training by incorporating physical laws into the loss function. Specifically, they use a continuous multilayer perceptron (MLP) to learn a probabilistic model of the data. The MLP is designed such that its output is constrained to satisfy certain physical laws or conservation principles. For example, in a simulation-based setting, the authors might require the predicted distribution to conserve mass-energy-momentum. This constraint is enforced by adding a penalty term to the loss function, which encourages the model to respect these physical constraints during training.

**CNN Surrogates (Li, 2020)**: In this approach, the authors

In [20]:
query_exam_tutor(
    "How does the constraint loss in BINNs enforce biological priors? Mention the specific mathematical constraints on Diffusivity.",
    system_context=my_custom_context
)


📝 Question: How does the constraint loss in BINNs enforce biological priors? Mention the specific mathematical constraints on Diffusivity.

🤖 Answer:
In Biologically-Informed Neural Networks (BINNs), the constraint loss, also known as the Biological Priors Loss, is used to enforce biologically meaningful constraints on the learned PDE parameters, specifically the diffusivity and growth rates.

The constraint loss is designed to penalize the model if the learned PDE parameters do not satisfy certain biological priors. In the case of BINNs, two specific mathematical constraints are enforced:

1. **Monotonicity**: The diffusivity must not decrease with increasing cell density (i.e., ∂D/∂u ≥ 0). This is based on the understanding that cells tend to move away from each other and thus have a higher diffusion rate when they are more sparse.
2. **Bounds**: Both the diffusivity and growth rates must remain within certain physiological bounds [min, max]. For example, the maximum growth rate may

In [21]:
query_exam_tutor(
    "Compare the loss functions and training data used in Lagergren and Li. How do these choices reflect the concepts of 'Physics-informed' vs 'Physics-based data'?",
    system_context=my_custom_context
)


📝 Question: Compare the loss functions and training data used in Lagergren and Li. How do these choices reflect the concepts of 'Physics-informed' vs 'Physics-based data'?

🤖 Answer:
**Loss Functions:**

1. **Lagergren (2020)**: The loss function consists of three components:
	* Generalized Least Squares (GLS) for handling heteroscedastic observation noise in scratch-assay data.
	* Physics Loss (PDE): Enforces the reaction-diffusion PDE residual via Automatic Differentiation to ensure physical plausibility.
	* Constraint Loss (Biological Priors): Penalizes if D or G exit bounds [min, max], or violate monotonicity constraints.

This multi-component loss function reflects the concept of **Physics-informed**, as it explicitly incorporates the governing equations (PDE) and biological priors to ensure that the model is physically plausible and consistent with the underlying biological processes.

2. **Li (2020)**: The loss function is simply the Mean Squared Error (MSE) between the CNN pre